In [ ]:
!pip install openai-whisper transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 6.1 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803980 sha256=e60cd415cd5fc62ad0f8e999285d98c7735f16fdb9e9698202f0fb9f1f706da5
  Stored in directory: /root/.cache/pip/wheels/ca/58/d5/fb4539ad74c3ca81eb40f7eda020ac77d080b33ad57449d485
Successfully built openai-whisper


In [ ]:
import whisper
model_whisper = whisper.load_model("base")

100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 126MiB/s]


In [ ]:
!wget -q https://groups.inf.ed.ac.uk/ami/AMICorpusMirror/amicorpus/ES2002a/audio/ES2002a.Mix-Headset.wav

In [ ]:
from pydub import AudioSegment

audio = AudioSegment.from_wav("ES2002a.Mix-Headset.wav")

sample = audio[:3 * 60 * 1000]

sample.export(
    "meeting_3min.wav",
    format="wav"
)

print("Done!")

Done!


/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
result = model_whisper.transcribe("meeting_3min.wav")

transcript = result["text"].strip()

print(transcript)

/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Um, like the ah. Uh, I'll put it in the back. Okay. Right. I know. Okay. Right. Well, it's the kickoff meeting for our project. Right. Well, it's the kickoff meeting for our project. And this is just what we're going to be doing over the next 25 minutes. So, first of all, just to kind of make sure that we all know each other. I'm Laura and I'm the project manager. Right. Do you want to introduce yourself again? David and I'm supposed to be a industrial designer. Okay. And I'm Andrew and I'm a marketing expert. I'm Greg and I'm a hugeurnt face. Great. Okay. So we're designing a new remote control. And, oh, I have to record he's here actually. So that's David, Andrew and Craig as much. And I'm going to be a new owner I've done today. And yes, we've designed a new remote control. As you can see, it's supposed to be original trendy and user friendly. So that's kind of our brief that we are. And so there are three different stages to the design. And I'm already sure what you guys have alrea

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
messages = [
    {
        "role": "user",
        "content": f"""
You are a meeting assistant.

Analyze the following meeting transcript and provide:

1. A short summary.
2. The key points discussed.
3. The action items or tasks mentioned.

Meeting transcript:
{transcript}
"""
    }
]

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True
)

input_ids = inputs["input_ids"]

In [ ]:
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=300
    )

In [ ]:
response = tokenizer.decode(
    outputs[0][input_ids.shape[-1]:],
    skip_special_tokens=True
)

print(response)

**Summary:**
The team has arrived at an initial stage of the remote control design process. They will receive feedback on their designs during three iterations (individual work, group meetings, and final trials). Each iteration involves drawing and summarizing favorite animals and attributes, allowing for iterative refinement of design concepts.

**Key Points Discussed:**
- The kickoff meeting was held to introduce everyone and outline the project scope.
- The team members were introduced, including their respective roles (project manager, industrial designer, and marketing expert).
- They started with individual design sessions where they could brainstorm and sketch ideas.
- Following these sessions, they went through a series of meetings to refine their designs.
- At the end of each session, participants could present their final drawings and summarize their favorites.

**Action Items/Mission:**
- Designers should continue refining their designs based on the feedback provided.
- Feed